Install Required Libraries

In [1]:

!pip -q install google-generativeai
!pip -q install sentence-transformers
!pip -q install faiss-cpu
!pip -q install pymupdf
!pip -q install pypdf
!pip -q install langchain
!pip -q install langchain-community
!pip -q install ipywidgets
!pip -q install fpdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Import Libraries

In [2]:


import os
import fitz
import faiss
import numpy as np

from google.colab import files
from google.colab import userdata

import google.generativeai as genai

from sentence_transformers import SentenceTransformer

from fpdf import FPDF

from IPython.display import display
import ipywidgets as widgets

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Connect Gemini API Securely

In [3]:
API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

print("Gemini Connected Successfully!")

Gemini Connected Successfully!


Test Gemini

In [4]:
response = model.generate_content(
    "Introduce yourself in one sentence."
)

print(response.text)

I am a large language model, trained by Google.


Create Project Memory

In [5]:
uploaded_papers = []

paper_metadata = []

paper_chunks = []

paper_embeddings = []

chat_history = []

search_history = []

print("Project memory initialized.")

Project memory initialized.


Load Embedding Model

In [6]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


Create Vector Database

In [7]:
dimension = 384

index = faiss.IndexFlatL2(dimension)

print("FAISS vector database created.")

FAISS vector database created.


Welcome Banner

In [8]:
print("=" * 60)
print("ResearchPilot AI")
print("Agentic AI Research Assistant")
print("=" * 60)

print("\nCurrent Status")
print("----------------------")
print("Gemini Connected")
print("Embedding Model Loaded")
print("Vector Database Ready")
print("Project Initialized")

ResearchPilot AI
Agentic AI Research Assistant

Current Status
----------------------
Gemini Connected
Embedding Model Loaded
Vector Database Ready
Project Initialized


PDF Upload

In [9]:
uploaded_files = files.upload()

print(f"\n{len(uploaded_files)} file(s) uploaded successfully.")

Saving Ethical_AI_Towards_Defining_a_Collective_Evaluation_Framework.pdf to Ethical_AI_Towards_Defining_a_Collective_Evaluation_Framework.pdf

1 file(s) uploaded successfully.


Function to Read PDF

In [38]:


def extract_pdf_text(pdf_path):

    document = fitz.open(pdf_path)

    text = ""

    for page in document:
        text += page.get_text()

    document.close()

    return text

Extract Metadata

In [39]:
import json

def extract_metadata(pdf_path):

    document = fitz.open(pdf_path)

    first_page = document.load_page(0).get_text()

    prompt = f"""
You are reading the first page of a research paper.

Extract the following information.

Return ONLY valid JSON.

{{
"title":"",
"authors":"",
"year":"",
"abstract":"",
"keywords":[]
}}

Paper:

{first_page}
"""

    try:

        response = model.generate_content(prompt)

        text = response.text.replace("```json", "").replace("```", "").strip()

        data = json.loads(text)

    except Exception as e:

        print(e)

        data = {
            "title": os.path.basename(pdf_path),
            "authors": "Unknown",
            "year": "Unknown",
            "abstract": "",
            "keywords": []
        }

    document.close()

    return data

Save Papers Into Memory

In [40]:
for filename in uploaded_files.keys():

    metadata = extract_metadata(filename)

    text = extract_pdf_text(filename)

    uploaded_papers.append({

        "title": metadata["title"],

        "author": metadata["authors"],

        "year": metadata["year"],

        "abstract": metadata["abstract"],

        "keywords": metadata["keywords"],

        "text": text

    })

print("Papers stored successfully.")

Papers stored successfully.


Display Uploaded Papers

In [43]:


print("=" * 80)
print("Uploaded Research Papers")
print("=" * 80)

for i, paper in enumerate(uploaded_papers):

    print(f"\n📄 Paper {i+1}")
    print(f"Title      : {paper.get('title', 'Unknown')}")
    print(f"Author     : {paper.get('author', 'Unknown')}")
    print(f"Year       : {paper.get('year', 'Unknown')}")

    # Handle keywords safely
    keywords = paper.get("keywords", [])

    if isinstance(keywords, list):
        keywords = ", ".join(keywords)
    elif keywords is None:
        keywords = "Not Available"

    print(f"Keywords   : {keywords}")

    print("\nAbstract:")
    print(paper.get("abstract", "Not Available"))

    print("-" * 80)

Uploaded Research Papers

📄 Paper 1
Title      : Ethical AI: Towards Defining a Collective Evaluation Framework
Author     : Unknown
Year       : Unknown
Keywords   : 

Abstract:
Not Available
--------------------------------------------------------------------------------

📄 Paper 2
Title      : Ethical AI: Towards Defining a Collective Evaluation Framework
Author     : ['Aasish Kumar Sharma', 'Dimitar Kyosev', 'Julian Kunkel']
Year       : 2025
Keywords   : Responsible AI, Ethical AI, Machine Ethics, AI Acts, Explainability, Ontology, Workflows, Transparency

Abstract:
Artificial Intelligence (AI) is transforming sectors such as healthcare, finance, and autonomous systems, offering powerful tools for innovation. Yet its rapid integration raises urgent ethical concerns related to data ownership, privacy, and systemic bias. Issues like opaque decision-making, misleading outputs, and unfair treatment in high-stakes domains underscore the need for transparent and accountable AI systems. 

Search by Title

In [15]:
def search_title(keyword):

    results = []

    keyword = keyword.lower()

    for paper in uploaded_papers:

        if keyword in paper["title"].lower():

            results.append(paper)

    return results
results = search_title("AI")

for paper in results:

    print(paper["title"])

Ethical AI: Towards Defining a Collective Evaluation Framework


Search by Author

In [16]:
def search_author(author_name):

    results = []

    author_name = author_name.lower()

    for paper in uploaded_papers:

        if author_name in paper["author"].lower():

            results.append(paper)

    return results

View Paper

In [19]:
def show_paper(index):

    paper = uploaded_papers[index]

    print("="*60)

    print("Title :", paper["title"])

    print("Author:", paper["author"])

    print("Pages :", paper["pages"])

    print("="*60)

    print(paper["text"][:3000])

In [20]:
show_paper(0)

Title : Ethical AI: Towards Defining a Collective Evaluation Framework
Author: Unknown
Pages : 6
Ethical AI: Towards Defining a Collective
Evaluation Framework
1st Aasish Kumar Sharma
Mathematics and Computer Science
Georg-August-Universit¨at G¨ottingen
G¨ottingen, Germany
aasish-kumar.sharma@gwdg.de,
orcid: 0000-0002-7514-2340
2nd Dimitar Kyosev
Legal Affairs
Alis Grave Nil Private Limited
Burgas, Bulgaria
kyosev.dimitar@gmail.com
3th Julian Kunkel
Mathematics and Computer Science
Georg-August-Universit¨at G¨ottingen
G¨ottingen, Germany
julian.kunkel@gwdg.de,
orcid: 0000-0002-6915-1179
Abstract—Artificial Intelligence (AI) is transforming sectors
such as healthcare, finance, and autonomous systems, offering
powerful tools for innovation. Yet its rapid integration raises
urgent ethical concerns related to data ownership, privacy, and
systemic bias. Issues like opaque decision-making, misleading
outputs, and unfair treatment in high-stakes domains underscore
the need for transparent and

Import Text Splitter

In [22]:
import langchain
print(langchain.__version__)
!pip install -q langchain-text-splitters

1.3.13


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [24]:


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

print("Text Splitter Ready")

Text Splitter Ready


Split Papers into Chunks

In [25]:
paper_chunks = []

for paper_id, paper in enumerate(uploaded_papers):

    chunks = text_splitter.split_text(paper["text"])

    for chunk_id, chunk in enumerate(chunks):

        paper_chunks.append({

            "paper_id": paper_id,

            "chunk_id": chunk_id,

            "title": paper["title"],

            "author": paper["author"],

            "text": chunk

        })

print(f"Total Chunks Created: {len(paper_chunks)}")

Total Chunks Created: 53


Generate Embeddings



In [26]:
embeddings = embedding_model.encode(
    [chunk["text"] for chunk in paper_chunks],
    show_progress_bar=True
)

embeddings = np.array(embeddings).astype("float32")

print("Embeddings Generated")

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings Generated


Store Embeddings in FAISS

In [27]:
index.add(embeddings)

print(f"{index.ntotal} vectors stored in FAISS")

53 vectors stored in FAISS


Semantic Search Function

In [28]:
def semantic_search(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for idx in indices[0]:

        results.append(paper_chunks[idx])

    return results

Test Semantic Search

In [29]:
results = semantic_search(
    "Ethical AI "
)

for result in results:

    print("=" * 80)

    print("Title :", result["title"])

    print("Author:", result["author"])

    print()

    print(result["text"][:500])

    print()

Title : Ethical AI: Towards Defining a Collective Evaluation Framework
Author: Unknown

cal implications of AI’s outcomes. AI systems, often operating
as ”black boxes”, generate decisions and recommendations
with far-reaching consequences. The absence of embedded
ethical considerations can result in biased, unfair, or even
harmful outcomes. For example:
• Biased algorithms in hiring or lending processes can
perpetuate societal inequalities.
• Unethical use of AI in surveillance can infringe on civil
liberties.
• Lack of explainability can erode trust, particularly in life-
critical

Title : Ethical AI: Towards Defining a Collective Evaluation Framework
Author: Unknown

The article is organized in five sections: Introduction (Sec-
tion I), background (Section II), literature review (Section III),
methodology and discussion (Section IV), then includes con-
clusion and recommendations (Section V).
II. BACKGROUND
Ethical AI refers to the development and deployment of
artificial intelligenc

Semantic Search with Scores

In [30]:
def semantic_search_with_score(query, top_k=5):

    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):

        results.append({

            "distance": float(distance),

            "title": paper_chunks[idx]["title"],

            "author": paper_chunks[idx]["author"],

            "text": paper_chunks[idx]["text"]

        })

    return results

Test with Scores

In [33]:
results = semantic_search_with_score(
    "Unethical use of AI in surveillance"
)

for r in results:

    print("=" * 80)

    print("Title :", r["title"])

    print("Author:", r["author"])

    print("Distance :", round(r["distance"], 3))

    print()

    print(r["text"][:300])

Title : Ethical AI: Towards Defining a Collective Evaluation Framework
Author: Unknown
Distance : 0.664

cal implications of AI’s outcomes. AI systems, often operating
as ”black boxes”, generate decisions and recommendations
with far-reaching consequences. The absence of embedded
ethical considerations can result in biased, unfair, or even
harmful outcomes. For example:
• Biased algorithms in hiring or
Title : Ethical AI: Towards Defining a Collective Evaluation Framework
Author: Unknown
Distance : 0.737

Establishing robust frameworks for ethical AI development
and deployment is imperative to ensure AI aligns with human
values and serves humanity equitably and responsibly.
A. Key Ethical Challenges in AI
• Data-Related Issues: These include questions around data
accuracy, inclusion of purposefully m
Title : Ethical AI: Towards Defining a Collective Evaluation Framework
Author: Unknown
Distance : 0.748

The article is organized in five sections: Introduction (Sec-
tion I), background (

In [34]:
def semantic_search_filtered(query, author=None, top_k=5):

    results = semantic_search(query, top_k * 3)

    if author:

        results = [

            r for r in results

            if author.lower() in r["author"].lower()

        ]

    return results[:top_k]

In [44]:
results = semantic_search_filtered(
    "Artificial Intelligence",
    author=""
)

for r in results:

    print(r["title"])

Ethical AI: Towards Defining a Collective Evaluation Framework
Ethical AI: Towards Defining a Collective Evaluation Framework
Ethical AI: Towards Defining a Collective Evaluation Framework
Ethical AI: Towards Defining a Collective Evaluation Framework
Ethical AI: Towards Defining a Collective Evaluation Framework


Multi-Agent Architecture

Agent Class

In [54]:
class Agent:

    def __init__(self, name, role):
        self.name = name
        self.role = role

    def execute(self, prompt, context=""):

        full_prompt = f"""
You are {self.name}.

Role:
{self.role}

Use ONLY the context below to answer.
If the answer is not present in the context, say:
"I couldn't find this information in the uploaded papers."

Context:
{context}

Question:
{prompt}
"""

        response = model.generate_content(full_prompt)

        return response.text

Create Agents

In [55]:
summary_agent = Agent(

    "Summary Agent",

    "Summarize research papers professionally."
)

qa_agent = Agent(

    "Question Answering Agent",

    "Answer questions using only the uploaded research papers."
)

search_agent = Agent(

    "Search Agent",

    "Find relevant research information."
)

gap_agent = Agent(

    "Research Gap Agent",

    "Identify limitations, research gaps and future work."
)

review_agent = Agent(

    "Literature Review Agent",

    "Generate a literature review using multiple papers."
)

print("All Agents Created Successfully")

All Agents Created Successfully


Test Summary Agent

In [58]:
context = uploaded_papers[0]["text"]

summary = summary_agent.execute(

    "Summarize the uploaded research paper.",

    context=context

)

print(summary)

This research paper proposes a modular ethical assessment framework for Artificial Intelligence (AI) systems, built on ontological blocks of meaning. The framework aims to address urgent ethical concerns arising from AI's rapid integration into sectors like healthcare, finance, and autonomous systems, including issues related to data ownership, privacy, systemic bias, opaque decision-making, misleading outputs, and unfair treatment.

**Key Challenges Addressed:**
*   **Data-Related Issues:** Questions around data ownership, consent, privacy, accuracy, and potential for data poisoning.
*   **Trustworthiness:** AI systems must be reliable, safe, and explainable to prevent phenomena like "hallucinations" and over-reliance on "black-box" outcomes.
*   **Ethical Implications of Outcomes:** The potential for biased, unfair, or harmful decisions, such as perpetuating societal inequalities in hiring or lending, infringing on civil liberties through surveillance, or eroding trust due to lack of

Test QA Agent

In [57]:
context = uploaded_papers[0]["text"]

answer = qa_agent.execute(

    "What are the ethical challenges discussed in the paper?",

    context=context

)

print(answer)

The ethical challenges discussed in the paper include:

*   **Data-Related Issues:** These encompass questions around data ownership, consent, privacy, misuse and exploitation, intellectual property rights, individual privacy, data accuracy, inclusion of purposefully misleading data, and data poisoning.
*   **Trustworthiness:** This involves ensuring AI systems are reliable, safe, and explainable to prevent risks such as "hallucinations" (misleading or incorrect outputs) and over-reliance on black-box outcomes. Maintaining detailed records of decision-making processes is also critical for fostering trust.
*   **Ethical Implications of Outcomes:** This refers to the potential for AI systems, often operating as "black boxes," to generate biased, unfair, or even harmful decisions and recommendations with far-reaching consequences. Examples include biased algorithms perpetuating societal inequalities (e.g., in hiring or lending), unethical use of AI infringing on civil liberties (e.g., sur

Test Research Gap Agent

In [60]:

context = uploaded_papers[0]["text"][:12000]

gap = gap_agent.execute(

    "Identify the research gaps and suggest future work based on this paper.",

    context=context

)

print(gap)

Based on the provided context, here are the identified research gaps and suggested future work:

**Research Gaps/Limitations:**

*   Challenges remain in automation and probabilistic reasoning within the proposed ontological block framework.
*   Implementation of Human-in-the-loop (HITL) AI approaches in Self Reinforcement Learning (SLR) remains limited.
*   Real-time, scalable use of explainable AI methods (e.g., SHAP, LIME) remains a key challenge in high-stakes clinical settings.
*   Ensuring real-time ethical decisions and system adaptability remains a core technical and ethical challenge in autonomous vehicles.
*   Implementing FAIR-aligned ontological blocks is constrained by the manual effort needed to build and maintain interoperable frameworks.
*   The current study relies on synthesizing literature, expert insights, and theoretical models rather than conducting empirical simulations, indicating a potential gap in empirical validation.

**Future Work:**

*   Further work is ne

Router Agent

In [61]:


def router(query):

    query = query.lower()

    if any(word in query for word in ["summary", "summarize"]):
        return summary_agent

    elif any(word in query for word in ["gap", "future", "limitation"]):
        return gap_agent

    elif any(word in query for word in ["review", "compare"]):
        return review_agent

    elif any(word in query for word in ["search", "find"]):
        return search_agent

    else:
        return qa_agent

Smart Chat

In [62]:
def chat(query):

    context = uploaded_papers[0]["text"][:12000]

    agent = router(query)

    print(f"\n Using: {agent.name}")
    print("-" * 60)

    answer = agent.execute(
        query,
        context=context
    )

    print(answer)

In [63]:
chat("Summarize the uploaded research paper.")


 Using: Summary Agent
------------------------------------------------------------
This research paper addresses the urgent ethical concerns arising from the rapid integration of Artificial Intelligence (AI) across sectors like healthcare, finance, and autonomous systems. These concerns include data ownership, privacy, systemic bias, opaque decision-making, misleading outputs (hallucinations), and unfair treatment.

To tackle these challenges, the article proposes a modular ethical assessment framework. This framework is built upon "ontological blocks of meaning," which are discrete, interpretable units encoding ethical principles such as fairness, accountability, and ownership. By integrating these blocks with FAIR (Findable, Accessible, Interoperable, Reusable) principles, the framework aims to support scalable, transparent, and legally aligned ethical evaluations, including compliance with regulations like the EU AI Act.

The paper demonstrates the framework using a real-world use 

Retrieve Relevant Chunks

In [64]:
def retrieve_context(query, top_k=5):

    results = semantic_search(query, top_k)

    context = ""

    sources = []

    for result in results:

        context += result["text"] + "\n\n"

        sources.append(result["title"])

    return context, list(set(sources))

Smart QA

In [65]:
def ask_research_assistant(question):

    context, sources = retrieve_context(question)

    agent = router(question)

    answer = agent.execute(

        question,

        context=context

    )

    print("=" * 70)

    print(f" Agent Used : {agent.name}")

    print("=" * 70)

    print(answer)

    print("\n Sources")

    for source in sources:

        print("•", source)

In [66]:
ask_research_assistant(

    "What are the ethical challenges discussed in the paper?"

)

 Agent Used : Question Answering Agent
The ethical challenges discussed in the paper are:
*   **Data-Related Issues:** These include concerns about data accuracy, the inclusion of purposefully misleading data, or even data poisoning.
*   **Trustworthiness:** AI systems need to be reliable, safe, and explainable to prevent risks such as hallucinations and over-reliance on black-box outcomes.
*   **Ethical Implications of Outcomes:** This refers to the potential for biased or harmful decisions, highlighting the necessity to embed ethical principles directly into AI systems.

 Sources
• Ethical AI: Towards Defining a Collective Evaluation Framework


In [67]:
ask_research_assistant(

    "Summarize the uploaded research paper."

)

 Agent Used : Summary Agent
The research paper "Ethical AI: Towards Defining a Collective Evaluation Framework" focuses on the ethical challenges posed by the rapid integration of Artificial Intelligence (AI) across various sectors like healthcare, finance, and autonomous systems.

The paper highlights that while AI offers powerful tools for innovation and significant benefits, its evolution introduces profound ethical concerns. These include issues related to data ownership, privacy, consent, and the ethical acquisition of the vast datasets used to train AI systems. Questions such as "Who owns the data?" and "Was it ethically obtained?" underscore potential risks of misuse and exploitation, impacting intellectual property rights and individual privacy.

Ethical AI, as defined in the paper, refers to the development and deployment of AI systems that prioritize fairness, transparency, accountability, and respect for fundamental rights. The goal of AI ethics is to promote safe and respon

Compare Papers

In [68]:


def compare_papers(index1, index2):

    paper1 = uploaded_papers[index1]

    paper2 = uploaded_papers[index2]

    prompt = f"""

Compare these research papers.

Paper 1

Title:
{paper1['title']}

Text:
{paper1['text'][:7000]}

Paper 2

Title:
{paper2['title']}

Text:
{paper2['text'][:7000]}

Compare:

• Objectives

• Methodology

• Results

• Limitations

• Future Work

"""

    response = model.generate_content(prompt)

    print(response.text)

In [69]:
compare_papers(0,0)

Based on the provided text for Paper 1 and Paper 2, it is evident that **both papers are identical**. The title, authors, affiliations, abstract, index terms, and the body text (up to the end of Section II) are precisely the same.

Therefore, the comparison for each category will yield the same points for both papers.

---

### Comparison of Research Papers

**Paper 1 and Paper 2 are identical based on the provided text.**

Here's a breakdown of their shared characteristics:

#### • Objectives

The primary objective of both papers is to **address urgent ethical concerns in AI by proposing a modular ethical assessment framework.** This framework is built on "ontological blocks of meaning" (discrete, interpretable units encoding ethical principles like fairness, accountability, ownership) and integrated with FAIR (Findable, Accessible, Interoperable, Reusable) principles. The goal is to support scalable, transparent, and legally aligned ethical evaluations, including compliance with regu

Research Idea Generator

In [70]:
def generate_research_ideas():

    paper = uploaded_papers[0]["text"][:12000]

    prompt = f"""

Based on this research paper,

suggest five innovative project ideas.

For each idea provide

1. Title

2. Description

3. Difficulty

4. Required Technologies

Paper

{paper}

"""

    response = model.generate_content(prompt)

    print(response.text)

In [71]:
generate_research_ideas()

The research paper "Ethical AI: Towards Defining a Collective Evaluation Framework" proposes a modular ethical assessment framework built on "ontological blocks of meaning" integrated with FAIR principles to achieve scalable, transparent, and legally aligned ethical evaluations for AI systems. It highlights key challenges such as data ownership, bias, black-box decision-making, and the need for explainability and accountability, especially in high-stakes domains like finance, healthcare, and autonomous vehicles.

Based on this, here are five innovative project ideas:

---

### Project Idea 1: Ontology-Assisted Ethical AI Framework Generator

1.  **Title:** Ethical AI Ontology & Framework Generator (EAOFG)

2.  **Description:** This project aims to automate and streamline the creation and adaptation of ethical AI assessment frameworks based on the paper's concept of ontological blocks. The EAOFG would provide an intuitive interface for users (ethicists, developers, policymakers) to defi

Literature Review Generator

In [72]:
def generate_literature_review():

    context = ""

    for paper in uploaded_papers:
        context += paper["text"][:6000] + "\n\n"

    prompt = f"""
Generate a professional literature review based on the following research papers.

Include:

1. Introduction

2. Existing Work

3. Research Gaps

4. Future Scope

Research Papers:

{context}
"""

    response = model.generate_content(prompt)

    print(response.text)

    return response.text

In [73]:
generate_literature_review()

## Literature Review: Towards a Collective Evaluation Framework for Ethical AI

### 1. Introduction

The rapid evolution and widespread integration of Artificial Intelligence (AI) across critical sectors like healthcare, finance, and autonomous systems have heralded transformative opportunities. However, this advancement is not without significant ethical complexities, presenting challenges ranging from data ownership and privacy to systemic bias and opaque decision-making. The pervasive "black-box" nature of many AI systems often leads to misleading outputs, unfair treatment, and decisions with far-reaching societal implications, such as perpetuating inequalities in hiring or lending, infringing on civil liberties through surveillance, or eroding trust in life-critical applications. Addressing these profound ethical dilemmas necessitates the development of robust, transparent, and accountable frameworks to ensure AI aligns with human values and serves humanity equitably.

The paper "E

'## Literature Review: Towards a Collective Evaluation Framework for Ethical AI\n\n### 1. Introduction\n\nThe rapid evolution and widespread integration of Artificial Intelligence (AI) across critical sectors like healthcare, finance, and autonomous systems have heralded transformative opportunities. However, this advancement is not without significant ethical complexities, presenting challenges ranging from data ownership and privacy to systemic bias and opaque decision-making. The pervasive "black-box" nature of many AI systems often leads to misleading outputs, unfair treatment, and decisions with far-reaching societal implications, such as perpetuating inequalities in hiring or lending, infringing on civil liberties through surveillance, or eroding trust in life-critical applications. Addressing these profound ethical dilemmas necessitates the development of robust, transparent, and accountable frameworks to ensure AI aligns with human values and serves humanity equitably.\n\nThe p

Flashcard Generator

In [74]:
def generate_flashcards():

    context = uploaded_papers[0]["text"][:12000]

    prompt = f"""
Create 10 flashcards from this research paper.

Format:

Q:
A:

Paper:

{context}
"""

    response = model.generate_content(prompt)

    print(response.text)

    return response.text

In [75]:
generate_flashcards()

Here are 10 flashcards based on the provided research paper:

Q: What are some urgent ethical concerns raised by the rapid integration of AI?
A: Data ownership, privacy, systemic bias, opaque decision-making, misleading outputs, and unfair treatment in high-stakes domains.

Q: What kind of framework does the article propose to address ethical AI challenges?
A: A modular ethical assessment framework built on ontological blocks of meaning.

Q: What are the two main components integrated into the proposed ethical assessment framework?
A: Ontological blocks of meaning and FAIR (Findable, Accessible, Interoperable, Reusable) principles.

Q: What are "ontological blocks of meaning" in the context of this paper?
A: Discrete, interpretable units that encode ethical principles such as fairness, accountability, and ownership.

Q: What do the FAIR principles stand for, and why are they relevant to ethical AI?
A: FAIR stands for Findable, Accessible, Interoperable, and Reusable. They support the o

'Here are 10 flashcards based on the provided research paper:\n\nQ: What are some urgent ethical concerns raised by the rapid integration of AI?\nA: Data ownership, privacy, systemic bias, opaque decision-making, misleading outputs, and unfair treatment in high-stakes domains.\n\nQ: What kind of framework does the article propose to address ethical AI challenges?\nA: A modular ethical assessment framework built on ontological blocks of meaning.\n\nQ: What are the two main components integrated into the proposed ethical assessment framework?\nA: Ontological blocks of meaning and FAIR (Findable, Accessible, Interoperable, Reusable) principles.\n\nQ: What are "ontological blocks of meaning" in the context of this paper?\nA: Discrete, interpretable units that encode ethical principles such as fairness, accountability, and ownership.\n\nQ: What do the FAIR principles stand for, and why are they relevant to ethical AI?\nA: FAIR stands for Findable, Accessible, Interoperable, and Reusable. Th

Quiz Generator

In [76]:
def generate_quiz():

    context = uploaded_papers[0]["text"][:12000]

    prompt = f"""
Generate 10 multiple-choice questions.

Each question should contain

Question

A

B

C

D

Correct Answer

Paper

{context}
"""

    response = model.generate_content(prompt)

    print(response.text)

    return response.text

In [77]:
generate_quiz()

Here are 10 multiple-choice questions based on the provided paper:

---

**Question 1**
What is the primary solution proposed by the article to address the urgent ethical concerns in AI?
A. Implementing strict government regulations on AI development
B. Developing a modular ethical assessment framework built on ontological blocks of meaning
C. Relying solely on Human-in-the-loop (HITL) AI approaches for oversight
D. Conducting extensive empirical simulations to identify AI biases
Correct Answer: B
Paper

**Question 2**
According to the article's introduction, which of the following is NOT listed as a key ethical challenge in AI?
A. Data-Related Issues, such as ownership and privacy
B. Trustworthiness, including issues like AI "hallucinations"
C. Ethical Implications of Outcomes, leading to biased decisions
D. Lack of computational power for advanced AI models
Correct Answer: D
Paper

**Question 3**
How does the article define "Ethical AI"?
A. AI systems that prioritize speed and effici

'Here are 10 multiple-choice questions based on the provided paper:\n\n---\n\n**Question 1**\nWhat is the primary solution proposed by the article to address the urgent ethical concerns in AI?\nA. Implementing strict government regulations on AI development\nB. Developing a modular ethical assessment framework built on ontological blocks of meaning\nC. Relying solely on Human-in-the-loop (HITL) AI approaches for oversight\nD. Conducting extensive empirical simulations to identify AI biases\nCorrect Answer: B\nPaper\n\n**Question 2**\nAccording to the article\'s introduction, which of the following is NOT listed as a key ethical challenge in AI?\nA. Data-Related Issues, such as ownership and privacy\nB. Trustworthiness, including issues like AI "hallucinations"\nC. Ethical Implications of Outcomes, leading to biased decisions\nD. Lack of computational power for advanced AI models\nCorrect Answer: D\nPaper\n\n**Question 3**\nHow does the article define "Ethical AI"?\nA. AI systems that p

Conversation Memory

In [78]:
conversation_memory = []

def ask_with_memory(question):

    context, sources = retrieve_context(question)

    history = "\n".join(conversation_memory[-5:])

    prompt = f"""
Conversation History

{history}

Context

{context}

Question

{question}
"""

    response = model.generate_content(prompt)

    conversation_memory.append(f"User: {question}")
    conversation_memory.append(f"Assistant: {response.text}")

    print(response.text)

In [79]:
ask_with_memory("Summarize the paper.")
ask_with_memory("Now tell me its future scope.")

It looks like you've provided a list of **references**, not a single paper to summarize. These references cover a range of topics related to Artificial Intelligence, including:

*   **AI Ethics and Safety:**
    *   The "trolley problem" in self-driving cars.
    *   General ethics of AI and guides for responsible design.
    *   Ethics by design for AI.
    *   Ontology-based approaches to engineering ethicality requirements.
    *   Ethical AI principles.
*   **Knowledge Representation and Ontologies:**
    *   OWL Web Ontology Language.
    *   Terminological and ontological analysis (e.g., NCI thesaurus).
*   **Explainable AI (XAI):**
    *   Review of machine learning interpretability methods.
*   **Reinforcement Learning:**
    *   Revisiting sparse rewards for goal-reaching.
*   **Responsible Research and Innovation (RRI):**
    *   The role of RRI in ethical AI governance.
    *   Exploring dimensions of responsible research systems.
    *   Institutionalizing RRI in the busine

Export Summary to PDF

In [80]:
from fpdf import FPDF

def export_summary_to_pdf(summary, filename="Research_Summary.pdf"):

    pdf = FPDF()

    pdf.add_page()

    pdf.set_font("Arial", size=12)

    pdf.multi_cell(0, 10, summary)

    pdf.output(filename)

    print(f"PDF saved as {filename}")

In [81]:
summary = generate_literature_review()

export_summary_to_pdf(summary)

## Literature Review: Towards Collective Evaluation Frameworks for Ethical AI

### 1. Introduction

The rapid advancement and pervasive integration of Artificial Intelligence (AI) across critical sectors such as healthcare, finance, and autonomous systems promise transformative benefits, including enhanced productivity and innovative solutions to global challenges. However, this accelerated adoption simultaneously introduces profound and urgent ethical dilemmas that demand rigorous attention. Core among these are concerns regarding data ownership, privacy, systemic biases embedded within algorithms, the opaqueness of AI decision-making processes, the generation of misleading outputs (hallucinations), and the potential for unfair treatment in high-stakes applications. The absence of embedded ethical considerations can lead to biased hiring practices, infringements on civil liberties through surveillance, and an erosion of trust due to a lack of explainability, particularly in life-criti

Research Dashboard

In [82]:
def research_dashboard():

    print("="*70)
    print("RESEARCHPILOT AI DASHBOARD")
    print("="*70)

    print(f"Total Papers Uploaded : {len(uploaded_papers)}")

    total_pages = 0

    for paper in uploaded_papers:

        if "pages" in paper:
            total_pages += paper["pages"]

    print(f"Total Pages           : {total_pages}")

    print(f"Vector Chunks         : {len(paper_chunks)}")

    print(f"Embeddings Stored     : {index.ntotal}")

    print("="*70)

In [83]:
research_dashboard()

RESEARCHPILOT AI DASHBOARD
Total Papers Uploaded : 2
Total Pages           : 6
Vector Chunks         : 53
Embeddings Stored     : 53


Paper Information

In [89]:
def paper_information():

    for i, paper in enumerate(uploaded_papers):

        print("\n" + "=" * 70)
        print(f"Paper {i+1}")
        print("=" * 70)

        print("Title      :", paper.get("title", "Unknown"))
        print("Author     :", paper.get("author", "Unknown"))
        print("Year       :", paper.get("year", "Unknown"))
        print("Keywords   :", paper.get("keywords", "Not Available"))
        print("Abstract   :", paper.get("abstract", "Not Available")[:300])

        print("=" * 70)

In [90]:
paper_information()


Paper 1
Title      : Ethical AI: Towards Defining a Collective Evaluation Framework
Author     : Unknown
Year       : Unknown
Keywords   : Not Available
Abstract   : Not Available

Paper 2
Title      : Ethical AI: Towards Defining a Collective Evaluation Framework
Author     : ['Aasish Kumar Sharma', 'Dimitar Kyosev', 'Julian Kunkel']
Year       : 2025
Keywords   : ['Responsible AI', 'Ethical AI', 'Machine Ethics', 'AI Acts', 'Explainability', 'Ontology', 'Workflows', 'Transparency']
Abstract   : Artificial Intelligence (AI) is transforming sectors such as healthcare, finance, and autonomous systems, offering powerful tools for innovation. Yet its rapid integration raises urgent ethical concerns related to data ownership, privacy, and systemic bias. Issues like opaque decision-making, mislea


AI Paper Categorizer

In [86]:
def categorize_paper():

    context = uploaded_papers[0]["text"][:8000]

    prompt = f"""
Read this research paper and classify it.

Choose one category.

Artificial Intelligence

Machine Learning

Deep Learning

Healthcare

Cyber Security

Computer Vision

Natural Language Processing

Finance

Robotics

Data Science

Paper

{context}
"""

    response = model.generate_content(prompt)

    print("Category")

    print(response.text)

Research Statistics

In [92]:
def research_statistics():

    print("="*70)

    print("Research Statistics")

    print("="*70)

    print("Uploaded Papers :", len(uploaded_papers))

    print("Chunks Created  :", len(paper_chunks))

    print("Vector Size     :", index.ntotal)

    print("Conversation    :", len(conversation_memory))

    print("="*70)

In [93]:
research_statistics()

Research Statistics
Uploaded Papers : 2
Chunks Created  : 53
Vector Size     : 53
Conversation    : 4


Main Menu

In [94]:
def menu():

    print("="*70)

    print("ResearchPilot AI")

    print("="*70)

    print("1 Dashboard")

    print("2 Ask Question")

    print("3 Literature Review")

    print("4 Flashcards")

    print("5 Quiz")

    print("6 Research Ideas")

    print("7 Compare Papers")

    print("8 Dashboard Statistics")

    print("="*70)

In [95]:
menu()

ResearchPilot AI
1 Dashboard
2 Ask Question
3 Literature Review
4 Flashcards
5 Quiz
6 Research Ideas
7 Compare Papers
8 Dashboard Statistics


Welcome Screen

In [96]:
print("""
========================================

🤖 ResearchPilot AI

Multi-Agent Research Assistant

Features

✅ Upload Research Papers

✅ Semantic Search

✅ RAG

✅ Multiple AI Agents

✅ Literature Review

✅ Flashcards

✅ Quiz Generator

✅ Research Gap Detection

✅ Research Idea Generator

========================================
""")



🤖 ResearchPilot AI

Multi-Agent Research Assistant

Features

✅ Upload Research Papers

✅ Semantic Search

✅ RAG

✅ Multiple AI Agents

✅ Literature Review

✅ Flashcards

✅ Quiz Generator

✅ Research Gap Detection

✅ Research Idea Generator


